In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import os, re, functools, json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

from src.scryfall_http_service import ScryfallHttpService
from src.scryfall_data_service import ScryfallDataService
from src.cache_service import CacheService, CacheType
from src import image_service

pd.set_option('display.max_columns', None)

os.getcwd()

'/Users/ptallo/Documents/mtg_data_science'

In [3]:
def get_commanders_from_txt_file(fpath: str) -> list[str]:
    with open(fpath, 'r') as f:
        return [x.strip() for x in f.read().split("\n")]

def get_json_file(fpath: str) -> dict:
    with open(fpath, 'r') as f:
        return json.loads(f.read())

In [4]:
sfds = ScryfallDataService(
    base_url="https://api.scryfall.com/", 
    json_cache_service=CacheService("./cache/json/", cache_type=CacheType.JSON), 
    png_cache_service=CacheService("./cache/png/", cache_type=CacheType.PNG), 
    http_client=ScryfallHttpService(),
)

# sfds.clear_caches()

In [5]:
cards_df = sfds.get_cards_from_bulk_data('gameplay_columns')
cards_df.shape

(38233, 23)

In [50]:
all_keywords_array = sorted(set([x.lower() for x in cards_df.keywords.explode().tolist() if isinstance(x, str)]))

In [14]:
def get_supertypes_from_type_line(type_line: str) -> list[str]:
    supertypes = ["basic", "legendary", "ongoing", "snow", "world"]
    return sorted([x for x in supertypes if x in type_line.lower()])

def get_types_from_type_line(type_line: str, valid_card_types: list[str]) -> list[str]:
    return sorted([x for x in valid_card_types if x.lower() in type_line.lower()])

def get_subtypes_from_type_line(type_line: str, type_to_subtype_dict: dict[str, list[str]]) -> list[str]:
    subtypes = []
    for k, v in type_to_subtype_dict.items():
        if k.lower() not in type_line.lower():
            continue
        for subtype in v:
            if subtype.lower() not in type_line.lower():
                continue
            subtypes.append(subtype.lower())
    return sorted(subtypes)

In [15]:
def drop_all_na_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_remove = []
    for c in df.columns:
        if df[~df[c].isna()].shape[0] > 0:
            continue 
        cols_to_remove.append(c)
    return df.drop(columns=cols_to_remove)

def rename_bool_cols(df: pd.DataFrame) -> pd.DataFrame:
    fix_col_name = lambda c: re.sub("^", "is_", c) if df[c].dtype == bool and len(re.findall("^is_", c)) == 0 else c
    df.columns = [fix_col_name(c) for c in df.columns]
    return df

In [16]:
def get_supertypes_from_type_line(type_line: str) -> list[str]:
    supertypes = ["basic", "legendary", "ongoing", "snow", "world"]
    return sorted([x for x in supertypes if x in type_line.lower()])

def get_types_from_type_line(type_line: str, valid_card_types: list[str]) -> list[str]:
    return sorted([x for x in valid_card_types if x.lower() in type_line.lower()])

def get_subtypes_from_type_line(type_line: str, type_to_subtype_dict: dict[str, list[str]]) -> list[str]:
    subtypes = []
    for k, v in type_to_subtype_dict.items():
        if k.lower() not in type_line.lower():
            continue
        for subtype in v:
            if subtype.lower() not in type_line.lower():
                continue
            subtypes.append(subtype.lower())
    return sorted(subtypes)

def parse_scryfall_data(df: pd.DataFrame, magic_metadata_dict: dict) -> pd.DataFrame:
    df['is_split_card'] = df['name'].apply(lambda x: "//" in x)
    df['supertypes'] = df['type_line'].apply(get_supertypes_from_type_line)
    df['types'] = df['type_line'].apply(functools.partial(get_types_from_type_line, valid_card_types=magic_metadata_dict.get("type_and_subtype_info").keys()))
    df['subtypes'] = df['type_line'].apply(functools.partial(get_subtypes_from_type_line, type_to_subtype_dict=magic_metadata_dict.get("type_and_subtype_info")))
    df['oracle_text_mod'] = df.oracle_text.apply(lambda x: x.split("\n"))
    return df

def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return df[[x for x in columns if x in df.columns]]

commander_df = cards_df[cards_df.is_commander_legal & ~cards_df.oracle_text.isna() & cards_df.keywords.str.len().ge(3)].copy() \
    .pipe(drop_all_na_columns) \
    .pipe(rename_bool_cols) \
    .pipe(functools.partial(parse_scryfall_data, magic_metadata_dict=get_json_file("./data/magic_metadata.json"))) \
    .pipe(functools.partial(select_columns, columns=['oracle_id', 'oracle_text_mod']))


,oracle_id,oracle_text_mod
195,01432542-2ea7-45bc-bc90-151516faee0e,"Flying, ward {4}"
195,01432542-2ea7-45bc-bc90-151516faee0e,Shrieking Gargoyles — Whenever this creature o...
244,018327d1-a3ba-4912-9c88-f0f0c54a1750,"Flying, lifelink"
244,018327d1-a3ba-4912-9c88-f0f0c54a1750,This creature can't block.
244,018327d1-a3ba-4912-9c88-f0f0c54a1750,Eerie — Whenever an enchantment you control en...


In [69]:
commander_df: pd.DataFrame = cards_df[
    cards_df.is_commander_legal & ~cards_df.oracle_text.isna() & 
    cards_df.keywords.str.len().ge(3)
].sample(frac=1).copy()
commander_df.keywords = commander_df.keywords.apply(lambda arr: [x.lower() for x in arr])
commander_df.oracle_text = commander_df.oracle_text.apply(lambda x: x.lower())
commander_df = commander_df[['oracle_text', 'keywords']]

def get_keywords_regex(keywords: list[str]) -> str:
    return r"(?:^|, ){1}(" + "|".join(keywords) + ")"

orr_iterable = zip(commander_df.oracle_text.tolist(), commander_df.keywords.tolist())
for text, keywords in [x for x in orr_iterable][:10]:
    print(keywords)

    abilities = text.split("\n")
    print(abilities)
    for a in abilities:
        matches = re.findall(get_keywords_regex(all_keywords_array), a)
        print("\t{}".format(matches))

    print("")

['amass', 'flashback', 'mill']
["amass orcs x. mill x cards. you may cast an instant or sorcery spell with mana value x or less from among them without paying its mana cost. (to amass orcs x, put x +1/+1 counters on an army you control. it's also an orc. if you don't control an army, create a 0/0 black orc army creature token first.)", 'flashback—{3}{u}{r}, exile x cards from your graveyard.']
	['amass']
	['flash']

['reach', 'vigilance', 'trample']
['vigilance, reach, trample', 'this creature gets +1/+1 as long as you control a desert.', 'this creature gets +1/+1 as long as there is a desert card in your graveyard.']
	['vigilance', 'reach', 'trample']
	[]
	[]

['echo', 'flying', 'haste']
['flying, haste', 'echo—discard a card. (at the beginning of your upkeep, if this came under your control since the beginning of your last upkeep, sacrifice it unless you pay its echo cost.)']
	['flying', 'haste']
	['echo']

['devoid', 'ingest', 'deathtouch']
['devoid (this card has no color.)', 'deat

In [62]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
commander_df = commander_df.explode(column=['oracle_text_mod']).head()
commander_df['embedding'] = commander_df.oracle_text_mod.apply(lambda x: model.encode(x))
commander_df.head()

,oracle_id,oracle_text_mod,embedding
195,01432542-2ea7-45bc-bc90-151516faee0e,"Flying, ward {4}","[0.026824947, 0.03292733, -0.05596602, -0.0235..."
195,01432542-2ea7-45bc-bc90-151516faee0e,Shrieking Gargoyles — Whenever this creature o...,"[-0.015568892, 0.02719046, 0.07449955, 0.03372..."
244,018327d1-a3ba-4912-9c88-f0f0c54a1750,"Flying, lifelink","[-0.036932345, -0.0016798618, -0.018819192, 0...."
244,018327d1-a3ba-4912-9c88-f0f0c54a1750,This creature can't block.,"[-0.02347612, 0.08012184, -0.027529735, 0.0027..."
244,018327d1-a3ba-4912-9c88-f0f0c54a1750,Eerie — Whenever an enchantment you control en...,"[-0.011237628, 0.0100720795, -0.045012873, 0.0..."
